# Fine-tune an Agentic Small Reasoning Model for Political Sciences.

This code notebook is an introduction to fine tuning a small reasoning model for political sciences and, more broadly, human and and social sciences. Most of the principles we'll see are applicable to all LLMs: the specificity of SRM is only on the data/instruction side, yet will real consequences over the model ulterior behavior.

There is no hard prerequisites in programming or artificial intelligence. LLM fine tuning is relatively accessible but still remains an emergent applied knowledge with little ressources tailored for people without a computer science background.

We are going to see in several steps:
* The fundamentals of LLM fine-tuning and its most common and economic approach, LORA.
* The design of training data and the increasing reliance on "synthetic" texts created by other models.
* The definition of training settings (also called "hyperparameters").
* Actual training.

Every step of this presentation is done by a Python script that does not need to be fully understood to proceed (and if needs be you can… further comment it using an LLM).

Fine-tuning will be done on a base gemma-3-4b model. The gemma 3 series from Google currently yield the best results for multilingual use cases.

This colab notebook can be run with an L4 GPU. Yet with the complete fine-tuning process it's preferable to use an A100, slightly less costly overall and much quicker.

## 1. Load the dataset.

We are going to finetune our LLM on a synthetic dataset. This is a very a common approach now in finetuning: there are very few available native *reasoning* dataset including both an instructions and a preliminary drafts and this would be very costly to make manually.

What we can do instead is using another, more powerful language model to generate an initial set that is then used for cleaning. Straight use of LLM generated data is usually termed *distillation* (though the name is a little confusing as it also includes other techniques of LLM-as-a-teacher). Yet, it is frequently common to reprocess the data afterwards, to remove potential issues, standardize the output or correct some unwanted behavior. This process is frequently called a *synthetic pipeline* or a *synthetic playground* and is increasingly becoming the preferred method to train reasoning models and agentic models due to the increased control over the data and the possibility to generate endlessly reasoning traces over actions that may not be very well documented.

We'll start by opening our dataset.

In [1]:
!wget https://github.com/Pclanglais/Parlementarian-Annotation/raw/refs/heads/main/discourse_generation_gemma.json.zip
!unzip discourse_generation_gemma.json.zip

--2025-09-04 19:36:14--  https://github.com/Pclanglais/Parlementarian-Annotation/raw/refs/heads/main/discourse_generation_gemma.json.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Pclanglais/Parlementarian-Annotation/refs/heads/main/discourse_generation_gemma.json.zip [following]
--2025-09-04 19:36:15--  https://raw.githubusercontent.com/Pclanglais/Parlementarian-Annotation/refs/heads/main/discourse_generation_gemma.json.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10455863 (10.0M) [application/zip]
Saving to: ‘discourse_generation_gemma.json.zip’

discourse_generatio 100%[==========

In [3]:
import pandas as pd

annotated_conversation = pd.read_json("discourse_generation_gemma.json")

annotated_conversation

,chunk_id,pivot_syceron,wikidata_label,wikidata_id,tag_entry,prediction,biography,completion,prediction_length,size_text,conversation
0,1167,3662856,Aurélien Le Coq,Q127256660,"<speech identifier=""3662812"" name=""Jean-Paul M...",C’est une simple possibilité !\n,Debating behavior: Confrontational\nStance: Ch...,### Analysis ###\nAurélien Le Coq's previous i...,10,Small,"### Debate ###\n<speech identifier=""3662812"" n..."
1,165,3679136,Xavier Breton,Q427080,"<speech identifier=""3679135"" name=""Éric Pauget...",Quel est l’avis du gouvernement ?\n,Debating behavior: Procedural\nStance: Facilit...,### Analysis ###\nThe provided features paint ...,10,Small,"### Debate ###\n<speech identifier=""3679135"" n..."
2,2135,3802682,Julie Laernoes,Q112650708,"<speech identifier=""3802672"" name=""Jean-Luc Fu...",Renouvelable !\n,Debating behavior: Confrontational\nDeliberati...,### Analysis ###\nThe context is a moment of a...,6,Small,"### Debate ###\n<speech identifier=""3802672"" n..."
3,1919,3574087,Éric Ciotti,Q1844180,"<speech identifier=""3573432"" name=""Éric Ciotti...",Toutes les collectivités ne sont pas d’accord....,Debating behavior: Confrontational\nArgumentat...,### Analysis ###\nThe debate presents a classi...,66,Medium,"### Debate ###\n<speech identifier=""3573432"" n..."
4,123,3690693,Cyrielle Chatelain,Q112648926,"<speech identifier=""3690681"" name=""Cyrielle Ch...","Il y aura, si le Rassemblement national le déc...",Emotion: Firm\nDeliberative quality: Justified...,### Analysis ###\nThe debate is highly content...,89,Medium,"### Debate ###\n<speech identifier=""3690681"" n..."
...,...,...,...,...,...,...,...,...,...,...,...
3499,1862,3729100,Émeric Salmon,Q63248686,"<speech identifier=""3728559"" name=""Julien Guib...","On est en démocratie, la ministre dit ce qu’el...",Debating behavior: Confrontational\nDebating b...,### Analysis ###\nThe session is extremely vol...,15,Small,"### Debate ###\n<speech identifier=""3728559"" n..."
3500,2993,3606621,Benjamin Lucas,Q26214575,"<speech identifier=""3606619"" name=""Paul Vannie...",Exactement !\n,Debating behavior: Confrontational\nArgumentat...,### Analysis ###\nThe provided speech by Paul ...,6,Small,"### Debate ###\n<speech identifier=""3606619"" n..."
3501,664,3642864,Matthias Tavel,Q50384786,"<speech identifier=""3642471"" name=""Sylvain Ber...",Ils ont réussi à hériter ! (Sourires sur les b...,Debating behavior: Confrontational\nStance: Ch...,### Analysis ###\nThe sequence of intervention...,17,Small,"### Debate ###\n<speech identifier=""3642471"" n..."
3502,292,3826257,Sabrina Sebaihi,Q112651173,"<speech identifier=""3826255"" name=""Agnès Panni...",Ils détricotent toutes les lois sur l’écologie...,Debating behavior: Confrontational\nArgumentat...,### Analysis ###\nThe preceding speech by Agnè...,33,Small,"### Debate ###\n<speech identifier=""3826255"" n..."


Let's take the first text to inspect the overall structure:

In [4]:
print(annotated_conversation["conversation"].tolist()[0])

### Debate ###
<speech identifier="3662812" name="Jean-Paul Mattei" group="Les Démocrates">
Nous en arrivons au cœur de ce texte. Je vous propose de rédiger différemment l’article 2 afin d’imposer aux établissements de crédit, dès la notification à leur client de la résiliation de sa convention de compte, de mentionner la possibilité pour ce dernier de saisir le médiateur de l’établissement. Ce médiateur, bien évidemment, ne sera pas libre de faire ce qu’il veut : il devra s’assurer, dans un délai d’un mois, que la résiliation intervient pour un motif légitime. Il appartient au législateur de caractériser ce motif légitime. Une résiliation serait ainsi abusive si elle était motivée par l’absence de rentabilité du compte ou la lourdeur administrative de la gestion de certains profils, comme celui de personnes politiquement exposées – même si, j’en conviens, on ne saurait légiférer pour les seules personnes politiquement exposées.Le médiateur devra également examiner les conséquences de 

As we can see this is a long text. What we have is:
* The general structure of an instruct qwen model (actually, originally llama), now repurposed for text annotations.
* The full debate with minimal contextual annotation (id syceron of speeches, date).
* The actual annotation using our preferred scheme.

For this initiation to fine-tuning, we didn't intend to run a long training. So keeping the data focused on the digital humanities subset made more sense.

## Setting up the fine-tuning.

At this point we are going to start the actual fine tuning process. I recommend to connect to a Google Drive to properly save the weights. Especially if you're finetuning for some time, there is always a risk of the training process being interrupted for whatever reason. Once we have a connection to the drive, we can make regular saves of the model weights.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir "/content/drive/My Drive/LLM"
%cd "/content/drive/My Drive/LLM"

Mounted at /content/drive
mkdir: cannot create directory ‘/content/drive/My Drive/LLM’: File exists
/content/drive/My Drive/LLM


Fine-tuning itself is relatively accessible.

For about the last two years, there has been a considerable effort dedicated to simplify, standardized and optimizer fine-tuning libraries. With some framework like Axolotl, it's not even necessary to program. Yet for this demo we're still going to use a python script which makes it easier to describe steps by steps, with a significant support from standardized libraries that do most of the work in the background: Transformers, Peft and Trl.

The current process of fine-tuning relies on three major concepts: weight freeze, matrix decomposition with LORA and quantization.

Originally model could only be *retrained*: the entire set of weights and parameters is changed to better predict a new corpus. This is now more commonly called *continuous pretraining* (in the sense that we are *continuing* the previous training). It's a common solution to significantly expand the ability of a model, especially to support a new human language. Yet it is costly and an entire cluster of GPUs is often required for this task.

With *fine-tuning* we **freeze the weights**: only a small part of the weights within the model are modified, leaving all the others untouched. This approach works rather well since fine tuning is actually done on small sample of data in comparison with the massive datasets used for pre-training. Weights freeze was first introduced a few years ago by Jeremy Howard and has repeatedly proven to be as effective as retraining. At the end of fine-tuning, the modified weights are simply *merged* on the original model.

Today the classic definition of *fine-tuning* has been rebranded as *full-finetuning* as it has been largely superseded by an even more effective approach: **LORA** or Low-Rank Adaptation. With we further break down of the unfrozen weights into two sub-matrices A and B. This is a standard form of matrix decomposition, commonly used in many other data compression tasks,  where the combination of two small matrices hold roughly the same representation and the same distribution as a large original matrices. With LORA it's also possible to keep the weights separated. This process is called a *LORA Adapter*. It is commonly used in image generation, less so for text as it has a negative impact on the speed of text generation. Currently mostly LORA finetunes still merge the weights.

Finally, to speed up the process even more, *fine-tuning* is usually done now on a quantized version of the LLM called **QLORA** (for *Quantized LORA*). Classically, each parameter in the weights was encoded in 32-bits. Repeated experiments showed that the precision of the parameters could be considerably decreased with little impact over the general performance. Today, most models are encoded by default in 16-bits. Quantization in 8-bits is commonplace as well. Starting with 4-bits we start to notice a loss of performance for text generation. And yet for fine-tuning, it turns out that training on a 4-bit model yields competitive results, provided the final LORA adapter si still merged in the end in a model with weights in 16-bit or 8-bit.

<img src="https://huggingface.co/datasets/probabl-ai/LLM_article/resolve/main/scheme_lora.png?download=true" width="400"/>

*Diagram of LORA fine-tuning: the unfrozen layers are decomposed into two matrices A and B. Qlora works nearly the same way, except that we load a quantized version of the model in 4-bit.*

We can start by loading all the necessary libraries. Lora (and LoraConfig) is the one managing the light fine-tuning on a limited amount of weights.

In [6]:
!pip install peft
!pip install trl
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 41.9 MB/s eta 0:00:00


Nous procédons à l'import des extensions :

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

Then we define the name of the base model and the name of the new model. By default the base model can be a folder containing the weights or the id of the model on HuggingFace (which will be imported unless it is not already available locally).

In [2]:
model_name = "unsloth/gemma-3-4b-pt"

torch.cuda.empty_cache()

new_model_name = "gemma-generation-parliament"
output_dir = "./gemma-generation-parliament"

Now we define the most critical parameters. They will largely affect the end result of the fine-tuning and are the one you are the most likely to tweak to get better results:
* *max_steps* is the number of individual steps during the fine-tuning. Basically it determines how long will your training goes. Through each training phase (also called an *epoch*) the model will see all the data.
* *learning_rate* is a specific value assessing how much will the model assimilate new knowledge contained in the fine-tuning dataset. Yet it is also a trade-off as the model will also forget old knowledge and capacities if the value is too high. I usually recommend to set lower values than the one used by default.
* *per_device_train_batch_size*: this is the number of examples sent through each step. This parameter is mostly affected by your available memory on the GPU. Otherwise, the more example you send, the quicker the fine-tuning will complete one epochs.
* *max_seq_length*: the context window, the number of words you can store on the LLM. It has a hard limit, coming from the original training, but afterwards you can lower it to get faster training.
* *save_steps*: a number of steps after which you will perform a save. It is practical for long training but can take up space on storage.
* *lr_scheduler_type*: how (and will) the learning rate *decay* and basically decreases at every step.

In [3]:
max_steps = 200 #For full training 12000. Keep a much lower value for test purposes.
per_device_train_batch_size = 1 #Number of batches to send per step. We use almost none: better for regularization and longer context training.
learning_rate = 3e-4 #The most important hyperparmater. We take a high value since we are tuning a base model.
max_seq_length = 4096 #Context window length. This eats *a lot* of vram With an A100 hard to push further.
save_steps = 1000 #Since we use a very small batch, steps are fast, no need to save them frequently.
lr_scheduler_type = "linear" #Learning rate scheduler. Better to decrease the learning rate for long training. I prefer linear over to cosine as it is more predictable: easier to restart training if needed.

Afterwards, I add a few more parameters that are beyond the scope of this tutorial and which are more rarely changed from their default values.

In [4]:
local_rank = -1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 1
max_grad_norm = 0.3
weight_decay = 0.01
lora_alpha = 16
lora_dropout = 0.1
lora_r = 64
group_by_length = True
num_train_epochs = 1
packing = False
gradient_checkpointing = True
optim = "paged_adamw_32bit"
warmup_ratio = 0.03
logging_steps = 1
device_map = {"": 0}
report_to = "tensorboard"

And finally we ensure that the model will be loaded in a quantized format with QLora. You can notice that we require using the original in 4-bit and modifying the parameters of further computation to deal with a 4-bit format.

In [5]:
use_4bit = True
use_nested_quant = False
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
num_train_epochs = 1
fp16 = True
bf16 = False

Then we define the **LORA** configuration. On top of passing the predefined hyperparameter we specify the weights we to modify in a list sent to target_modules. Here for an initial training we use the lighter configuration with only two series of weights called **q_proj** and **v_proj**.

In [7]:
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    inference_mode=False,
    task_type="CAUSAL_LM",
    target_modules = ["q_proj", "v_proj"]
)

For a deeper fine-tune, that will take up more VRAM and processing time, you can also decide to select all the available weights to unfreeze. I would not recommend to use that on a first run as it's way better to first test your general model/data design and only assess afterwards if it's worth it to go through a longer training:

In [6]:
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    inference_mode=False,
    task_type="CAUSAL_LM",
    target_modules = ["q_proj", "v_proj", "k_proj", "gate_proj", "up_proj", "down_proj"]
)

We are now ready to load all the necessary components for the fine-tuning. We start with the tokenizer, basically a dataset that will map words or sub-words to their corresponding ids in the model.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


We can now import the dataset as a file. Since we already defined a consistent scheme for the instruct, we don't need to change anything.

In [9]:
!wget https://github.com/Pclanglais/Parlementarian-Annotation/raw/refs/heads/main/discourse_generation_gemma.json.zip
!unzip discourse_generation_gemma.json.zip

--2025-09-04 19:41:06--  https://github.com/Pclanglais/Parlementarian-Annotation/raw/refs/heads/main/discourse_generation_gemma.json.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Pclanglais/Parlementarian-Annotation/refs/heads/main/discourse_generation_gemma.json.zip [following]
--2025-09-04 19:41:07--  https://raw.githubusercontent.com/Pclanglais/Parlementarian-Annotation/refs/heads/main/discourse_generation_gemma.json.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10455863 (10.0M) [application/zip]
Saving to: ‘discourse_generation_gemma.json.zip.1’

discourse_generatio 100%[========

In [8]:
from datasets import load_dataset

def format_dataset(sample):
    prompt = f"{sample['conversation']}"
    return prompt

def template_dataset(sample):
    sample["text"] = f"{format_dataset(sample)}{tokenizer.eos_token}"
    return sample

data_files = {"train": "discourse_generation_gemma.json"}
dataset = load_dataset("json", data_files=data_files, split="train")

In [9]:
dataset

Dataset({
    features: ['chunk_id', 'pivot_syceron', 'wikidata_label', 'wikidata_id', 'tag_entry', 'prediction', 'biography', 'completion', 'prediction_length', 'size_text', 'conversation'],
    num_rows: 3504
})

And we load the model. Notice that we have to pass all the parameters for Qlora with 4-bit quantization. Overall this should take a few minutes for 4B. If all goes well you can notice that the model now hold up some vram on your graphic card (with nvidia, you can check with nvidia-smi).

In [10]:
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16, you can accelerate training with the argument --bf16")
        print("=" * 80)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device_map,
    quantization_config=bnb_config
)

model.config.use_cache = False
model.config.pretraining_tp = 1

torch.cuda.empty_cache()

Your GPU supports bfloat16, you can accelerate training with the argument --bf16


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

At this point, we can launch the actual training. Again not much to do since we are just passing already defined parameters. After executing this part of the code, your vram will be further filled by loading all the necessary components for fine-uning and, if all goes well, you will start seeing the logging messages of fine-tuning being printed regularly.

In [ ]:
from trl import SFTConfig

def formatting_func(example):
    return example["conversation"]  # Your data is already properly formatted

training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    gradient_checkpointing=True,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    max_length=max_seq_length,
    packing=packing
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments,
    formatting_func=formatting_func
)

#Training:
trainer.train()

Truncating train dataset:   0%|          | 0/3504 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Step,Training Loss
1,1.888000
2,1.617900
3,1.925400
4,1.809800
5,1.837700
6,2.030100
7,1.938800
8,2.074500
9,1.843900
10,2.082800


Overall, fine-tuning takes some time to heat up: the first step will take longer and you shouldn't care too much about the initial time estimate for the full training. After a dozen steps or so, training will find its standard pace.

Over time you should notice the following evolutions:
* The *loss* will generally go down. This is the main measure of the efficiency of fine-tuning: basically through each step, the model is rating itself by comparing a text generation with the original text in the dataset. The lower the loss, the better the grade. In my experience, the absolute value of the loss is not that indicative. The more important part is to see it going down regularly, as it shows that the model continues training and assimilate new knowledge. If the learning rate is too high you may notice a strong decrease at first and then a plateau, as your model has basically overfitted and will very quickly assimilate the more formal aspect of the data and miss on more subtle ones. Although as we have set only one batch to hold in the available memory, you will notice significant swings across the loss: the model is training on each indivvidual text successively and some of theses texts will be more challenging or unusual than others (usually the ones in multiple languages).
* The *learning rate* will go down as well if you opted for a "linear" or a "cosine" schedule. You may notice that the learning rate will actually go up initially in the *heating* phase before startin its slow continuous decline.

If you run into an issue or if you realize you may have undertrained the model, it's possible to continue training by relaunching the code and simply modifiying the trainer command like this:

In [ ]:
trainer.train(resume_from_checkpoint=True)

For this I recommend to set scheduling of the learning rate to "linear" or to "constant". "cosine" learning are harder to resume correctly.

Once the training is over, we can merge the weights. As you will notice in the directory you have set, for now you only have the LORA adapter, that contains the modified weights. While it can be used in itself for text generation, this is not really recommended as it can have a significant negative impact over generation speed. Instead we will merge the LORA on the entire original weights.

Also notice that we reload the weights in their original unquantized formats rather than 4-bits. Basically we're taking advantage of the "breadth" of the original model in terms of knowledge and accuracy, while still retaking the transformative matrix we got from fine-tuning:

In [ ]:
model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
model_to_save.save_pretrained(new_model_name)

torch.cuda.empty_cache()

from peft import AutoPeftModelForCausalLM
import os

model = AutoPeftModelForCausalLM.from_pretrained(new_model_name, device_map="auto", torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

output_merged_dir = os.path.join(new_model_name, new_model_name)
model.save_pretrained(output_merged_dir, safe_serialization=True)

#We also save the tokenizer
tokenizer.save_pretrained(output_merged_dir)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

('qwen-annotation-parliament/qwen-annotation-parliament/tokenizer_config.json',
 'qwen-annotation-parliament/qwen-annotation-parliament/special_tokens_map.json',
 'qwen-annotation-parliament/qwen-annotation-parliament/chat_template.jinja',
 'qwen-annotation-parliament/qwen-annotation-parliament/vocab.json',
 'qwen-annotation-parliament/qwen-annotation-parliament/merges.txt',
 'qwen-annotation-parliament/qwen-annotation-parliament/added_tokens.json',
 'qwen-annotation-parliament/qwen-annotation-parliament/tokenizer.json')

## Test the model

At the end of th fine-tuning phase we are able to test the model using model.generate.

It's better to start with a fresh session

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/My Drive/LLM"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/LLM


And reimport with all the necessary imports.

In [ ]:
import os, re
import pandas as pd
from pprint import pprint
import argparse
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    LlamaTokenizerFast
)
import copy
from pathlib import Path

Nous importons directement le modèle depuis notre sauvegarde locale.



In [ ]:
model = AutoModelForCausalLM.from_pretrained("/content/qwen-annotation-parliament/qwen-annotation-parliament")
tokenizer = AutoTokenizer.from_pretrained("/content/qwen-annotation-parliament/qwen-annotation-parliament")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Set the device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): 

And we define the prompt.

In [ ]:
text = """<|im_start|>user
French parliament session from 2025-10-22

General topic: Questions au Gouvernement > Fine-tuning de modèles de langue

Specific topic:Questions au Gouvernement > Fine-tuning de modèles de langue

Speech n°3521830 from Jean-Édouard Truc (Representative of the republican right) :

Enfin, nous nous appliquons maintenant à adapter des modèles de langue pour simplifier la retranscription des débats à l'assemblée. C'est, je crois, une tâche importante pour la souveraineté du pays.

Speech n°3521832 from Jules Machin (Representative of the far left) :

Sauf erreur de ma part, cher collègue, votre choix s'est porté sur Qwen qui est un modèle chinois. Pour la souveraineté nous repasserons.

Speech n°3521830 from Jean-Édouard Truc (Representative of the republican right) :

Cher collègue, avec tous le respect que je vous dois, je crois que vous n'y connaissez vraiment que dalle.
<|im_end|>
"""

inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
        ).to(device)

outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=1000,
            repetition_penalty=1,
            temperature=.3) #Low temperature since the model is weakly trained.

output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(output)

user
French parliament session from 2025-10-22

General topic: Questions au Gouvernement > Fine-tuning de modèles de langue

Specific topic:Questions au Gouvernement > Fine-tuning de modèles de langue

Speech n°3521830 from Jean-Édouard Truc (Representative of the republican right) :

Enfin, nous nous appliquons maintenant à adapter des modèles de langue pour simplifier la retranscription des débats à l'assemblée. C'est, je crois, une tâche importante pour la souveraineté du pays.

Speech n°3521832 from Jules Machin (Representative of the far left) :

Sauf erreur de ma part, cher collègue, votre choix s'est porté sur Qwen qui est un modèle chinois. Pour la souveraineté nous repasserons.

Speech n°3521830 from Jean-Édouard Truc (Representative of the republican right) :

Cher collègue, avec tous le respect que je vous dois, je crois que vous n'y connaissez vraiment que dalle.

assistant
<proposal>
<proposal_name>Adapting Language Models for Assembly Debates</proposal_name>
<proposal_nam

Overall due to the weak training the data structure should not be fully assimilated yet, though still sufficiently to see the quick effect of fine-tuning.